In [ ]:
import numpy  as np
import xarray as xr
import constants
import gc
from decomposition import coriolis_parameter,read_averages,average_averages
from energy_from_u import N_from_u
import scipy.interpolate as interpolate
import multiprocessing as mp
from wave_action import read_decomposition_averages_plusminus, intrinsic_frequency_alltri, coriolis_parameter, compute_m_nan, group_velocity_vertical, wave_numbers, coarse_grain

In [ ]:
file_slide = 0
levth=np.array([453.208, 400.000, 400.000, 400.000, 400.000, 373.779])[[0]] #Thickness of the levels in the ICON simulation
nrproc = 128  # Number of processes to be run in parallel
dir_dec = f"/work/bb1097/b383521/GW_Data_Summer/decomposition/slide_{file_slide}/"
save_dir = "/work/bb1097/b383521/GW_Data_Summer/coarsed_action/"
filename_beg="sp3D-gw_"
filename_mid = "-ny4_window_group-"
filename_end = ".nc"
av_dir  = "/work/bb1097/b383521/resolved_var/"
av_filename_beg = "average_"
av_filename_end = "-sphere_2016.nc"
for groupnr in [7]:
     print(f"groupnr {groupnr}")
     ampls_plus,ampls_minus,k,l,omega,maxind,ntri, av_var, ti= read_decomposition_averages_plusminus(dir_dec, filename_beg,filename_mid, filename_end, groupnr, av_dir, av_filename_beg, av_filename_end, file_slide=file_slide)
     freqs_alltri=dict(zip(["k",'l','omega'],[k,l,omega]))
     omega_hat=	intrinsic_frequency_alltri(freqs_alltri,av_var['u'],av_var['v'])
     ampls_plus[omega_hat<0]=np.nan #Should be used if taking both plus and minus. (1, 10, 143, 17, 23)
     ampls_minus[omega_hat<0]=np.nan
     path_grid = "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
     f=coriolis_parameter(xr.open_dataset(path_grid)['clat'].data)[ntri]
     m_plus=compute_m_nan(f,av_var,k,l,omega_hat,ampls_plus,levth)
     N_wadensity_minus = N_from_u(omega_hat,k,l,f,av_var,np.abs(ampls_minus))
     cgz = group_velocity_vertical(av_var, k, l, f, m_plus)
     m_target, l_target, k_target = wave_numbers()
     print("k_target (rad/m):", k_target)
     print("k_target → lambda_h (km):", 2*np.pi / np.abs(k_target[4:]) / 1e3)
     print("m_target (rad/m):", m_target)
     print("m_target → lambda_z (km):", 2*np.pi / m_target / 1e3)
     N_coarse_grain = coarse_grain(k_target, l_target, m_target, m_plus, k, l , N_wadensity_minus, cgz, ti)

def linear_edges(centers):
    """Midpoint edges for sorted array, works for negative values."""
    inner = 0.5 * (centers[:-1] + centers[1:])
    left  = centers[0]  - 0.5 * (centers[1]  - centers[0])
    right = centers[-1] + 0.5 * (centers[-1] - centers[-2])
    return np.concatenate([[left], inner, [right]])

def log_edges(centers):
    """Geometric mean edges for positive log-spaced centers."""
    log_c = np.log(centers)
    inner = np.exp(0.5 * (log_c[:-1] + log_c[1:]))
    left  = centers[0]  ** 2 / inner[0]
    right = centers[-1] ** 2 / inner[-1]
    return np.concatenate([[left], inner, [right]])
   
N_cgz = (cgz * N_wadensity_minus.real)
k_edges = linear_edges(k_target)   # (9,)  for 8 bins
l_edges = linear_edges(l_target)   # (9,)  for 8 bins
m_edges = log_edges(m_target)      # (5,)  for 4 bins

print("\nk_edges (rad/m):", k_edges)
print("l_edges (rad/m):", l_edges)
print("m_edges (rad/m):", m_edges)

# ── 5. Broadcast k, l to full shape ───────────────────────────────────────
k_3d  = k[np.newaxis, :, np.newaxis, np.newaxis, :]
l_3d  = l[np.newaxis, :, np.newaxis, : ,np.newaxis]
k_full = np.broadcast_to(k_3d, m_plus.shape)   # (2,128,144,23,23)
l_full = np.broadcast_to(l_3d, m_plus.shape)   # (2,128,144,23,2)
     
R_action = (
    np.nansum(N_cgz, axis=(2, 3, 4))
    / np.nansum(N_wadensity_minus.real, axis=(2, 3, 4))
)

print("Median action-weighted cgz:", np.nanmedian(R_action))
print("Mean action-weighted cgz:", np.nanmean(R_action))

R_action = (
    np.nansum(N_coarse_grain, axis=(2, 3, 4))
    / np.nansum(N_wadensity_minus.real, axis=(2, 3, 4))
)
print("Median action-weighted cgz:", np.nanmedian(R_action))
print("Mean action-weighted cgz:", np.nanmean(R_action))

In [ ]:
def linear_edges(centers):
    """Midpoint edges for sorted array, works for negative values."""
    inner = 0.5 * (centers[:-1] + centers[1:])
    left  = centers[0]  - 0.5 * (centers[1]  - centers[0])
    right = centers[-1] + 0.5 * (centers[-1] - centers[-2])
    return np.concatenate([[left], inner, [right]])

def log_edges(centers):
    """Geometric mean edges for positive log-spaced centers."""
    log_c = np.log(centers)
    inner = np.exp(0.5 * (log_c[:-1] + log_c[1:]))
    left  = centers[0]  ** 2 / inner[0]
    right = centers[-1] ** 2 / inner[-1]
    return np.concatenate([[left], inner, [right]])
   
N_cgz = (cgz * N_wadensity_minus.real)
k_edges = linear_edges(k_target)   # (9,)  for 8 bins
l_edges = linear_edges(l_target)   # (9,)  for 8 bins
m_edges = log_edges(m_target)      # (5,)  for 4 bins

print("\nk_edges (rad/m):", k_edges)
print("l_edges (rad/m):", l_edges)
print("m_edges (rad/m):", m_edges)

# ── 5. Broadcast k, l to full shape ───────────────────────────────────────
k_3d  = k[np.newaxis, :, np.newaxis, np.newaxis, :]
l_3d  = l[np.newaxis, :, np.newaxis, : ,np.newaxis]
k_full = np.broadcast_to(k_3d, m_plus.shape)   # (2,128,144,23,23)
l_full = np.broadcast_to(l_3d, m_plus.shape)   # (2,128,144,23,2)

In [ ]:
R_action = (
    np.nansum(N_cgz, axis=(2, 3, 4))
    / np.nansum(N_wadensity_minus.real, axis=(2, 3, 4))
)

print("Median action-weighted cgz:", np.nanmedian(R_action))
print("Mean action-weighted cgz:", np.nanmean(R_action))

R_action = (
    np.nansum(N_coarse_grain, axis=(2, 3, 4))
    / np.nansum(N_wadensity_minus.real, axis=(2, 3, 4))
)
print("Median action-weighted cgz:", np.nanmedian(R_action))
print("Mean action-weighted cgz:", np.nanmean(R_action))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

t, tri = 0, 0

k_edges = linear_edges(k_target)
l_edges = linear_edges(l_target)
m_edges = log_edges(m_target)

N  = np.real(N_cgz[t, tri])          # (omega, l, k)
M  = np.real(m_plus[t, tri])         # (omega, l, k)
kk, ll = k[tri], l[tri]              # native axes of this triangle

nm = len(m_target)
before, after = [], []
for im in range(nm):
    mask = np.isfinite(M) & np.isfinite(N) & (M >= m_edges[im]) & (M < m_edges[im+1])
    f = np.where(mask, N, 0.0).sum(axis=0)       # sum over omega -> (l, k)
    f[mask.sum(axis=0) == 0] = np.nan            # empty cells stay blank
    before.append(np.log1p(f))
    after.append(np.log1p(N_coarse_grain[t, tri, :, :, im].T))   # (l, k)

vmax = np.nanmax([np.nanmax(a) for a in before + after])
norm = Normalize(vmin=0, vmax=vmax)

fig, ax = plt.subplots(2, nm, figsize=(2.4*nm, 5), sharex=True, sharey=True,
                       constrained_layout=True)
for im in range(nm):
    lam = 2*np.pi / m_target[im] / 1e3
    ax[0, im].pcolormesh(kk*1e4, ll*1e4, before[im], norm=norm,
                         cmap="viridis", shading="nearest")
    pc = ax[1, im].pcolormesh(k_edges*1e4, l_edges*1e4, after[im], norm=norm,
                              cmap="viridis", shading="flat")
    ax[0, im].set_title(rf"$\lambda_z={lam:.1f}$ km")
    ax[1, im].set_xlabel(r"$k\ [10^{-4}\,\mathrm{m^{-1}}]$")
ax[0, 0].set_ylabel(r"before" "\n" r"$\ell\ [10^{-4}\,\mathrm{m^{-1}}]$")
ax[1, 0].set_ylabel(r"after" "\n" r"$\ell\ [10^{-4}\,\mathrm{m^{-1}}]$")
fig.colorbar(pc, ax=ax, label=r"$\log(1+\mathcal{N}c_{g,z})$")
fig.savefig("coarse_graining_before_after.pdf", dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec


def plot_before_after_coarsegrain(
    N, M, k_native, l_native,          # N, M: (omega, l, k) for one triangle/time; k_native, l_native: 1D native axes
    N_coarse,                           # (nk, nl, nm) coarse-grained field for the same triangle/time
    k_edges, l_edges, m_edges,          # coarse edges (rad/m)
    savename,
    n_sub=2,                            # fine sub-bands per coarse m bin
    cmap="viridis",
    fontsize=11,
    n_ticks=3,
):
    nm_c = len(m_edges) - 1
    n_top = nm_c * n_sub

    # fine m edges: split each coarse log-bin geometrically, so the edges nest
    fine_edges = np.concatenate([
        np.geomspace(m_edges[i], m_edges[i + 1], n_sub + 1)[:-1] for i in range(nm_c)
    ] + [[m_edges[-1]]])

    # --- top row: native k-l grid, summed over omega within each fine m band ---
    top = []
    for i in range(n_top):
        mask = np.isfinite(M) & np.isfinite(N) & (M >= fine_edges[i]) & (M < fine_edges[i + 1])
        f = np.where(mask, N, 0.0).sum(axis=0)           # (l, k)
        f[mask.sum(axis=0) == 0] = np.nan                # empty cells stay blank
        top.append(np.log1p(f))

    # --- bottom row: coarse bins ---
    bottom = [np.log1p(N_coarse[:, :, j].T) for j in range(nm_c)]   # (l, k)

    vmax = np.nanmax([np.nanmax(a) for a in top + bottom])
    norm = Normalize(vmin=0, vmax=vmax)

    # identical extent in both rows -> identical panel size
    ext = [k_edges[0], k_edges[-1], l_edges[0], l_edges[-1]]
    dk, dl = np.median(np.diff(k_native)), np.median(np.diff(l_native))
    ext_native = [k_native.min() - dk / 2, k_native.max() + dk / 2,
                  l_native.min() - dl / 2, l_native.max() + dl / 2]

    fig = plt.figure(figsize=(1.0 * n_top + 1.0, 3.6), constrained_layout=True)
    gs = GridSpec(2, n_top, figure=fig)
    offset = (n_top - nm_c) // 2                         # centers the bottom row

    def style(ax, first_col, bottom_row):
        ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
        ax.set_aspect("equal")
        ax.axhline(0, color="k", lw=0.5, alpha=0.3)
        ax.axvline(0, color="k", lw=0.5, alpha=0.3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{int(round(x*1e4))}"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, p: f"{int(round(y*1e4))}"))
        ax.tick_params(labelsize=fontsize - 2)
        if not first_col: ax.set_yticklabels([])
        if not bottom_row: ax.set_xticklabels([])

    for i in range(n_top):
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(top[i], origin="lower", extent=ext_native,
                       interpolation="nearest", cmap=cmap, norm=norm)
        lam = 2 * np.pi / np.sqrt(fine_edges[i] * fine_edges[i + 1]) / 1e3
        ax.set_title(rf"${lam:.1f}$ km", fontsize=fontsize - 2, pad=4)
        style(ax, i == 0, False)
        if i == 0: ax.set_ylabel("Before", fontsize=fontsize)

    for j in range(nm_c):
        ax = fig.add_subplot(gs[1, offset + j])
        ax.imshow(bottom[j], origin="lower", extent=ext,
                  interpolation="nearest", cmap=cmap, norm=norm)
        lam = 2 * np.pi / np.sqrt(m_edges[j] * m_edges[j + 1]) / 1e3
        ax.set_title(rf"$\lambda_z={lam:.1f}$ km", fontsize=fontsize - 2, pad=4)
        style(ax, j == 0, True)
        if j == 0: ax.set_ylabel("After", fontsize=fontsize)

    cbar = fig.colorbar(im, ax=fig.axes, fraction=0.02, pad=0.01)
    cbar.set_label(r"$\log(1+\mathcal{N})$", fontsize=fontsize)
    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    for ext_ in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext_}",
                    dpi=400, bbox_inches="tight")
    plt.show()


# usage
t, tri = 0, 0
plot_before_after_coarsegrain(
    np.real(N_cgz[t, tri]), np.real(m_plus[t, tri]), k[tri], l[tri],
    N_coarse_grain[t, tri], k_edges, l_edges, m_edges,
    savename="coarse_graining_before_after", n_sub=2,
)

In [ ]:
def plot_before_after_coarsegrain(
    N, M, k_native, l_native,
    N_coarse,
    k_edges, l_edges, m_edges,
    savename,
    n_sub=2,
    cmap="viridis",
    fontsize=11,
):
    nm_c = len(m_edges) - 1
    n_top = nm_c * n_sub

    fine_edges = np.concatenate([
        np.geomspace(m_edges[i], m_edges[i + 1], n_sub + 1)[:-1] for i in range(nm_c)
    ] + [[m_edges[-1]]])

    top = []
    for i in range(n_top):
        mask = np.isfinite(M) & np.isfinite(N) & (M >= fine_edges[i]) & (M < fine_edges[i + 1])
        f = np.where(mask, N, 0.0).sum(axis=0)
        f[mask.sum(axis=0) == 0] = np.nan
        top.append(np.log1p(f))
    bottom = [np.log1p(N_coarse[:, :, j].T) for j in range(nm_c)]

    vmax = np.nanmax([np.nanmax(a) for a in top + bottom])
    norm = Normalize(vmin=0, vmax=vmax)

    ext = [k_edges[0], k_edges[-1], l_edges[0], l_edges[-1]]
    dk, dl = np.median(np.diff(k_native)), np.median(np.diff(l_native))
    ext_native = [k_native.min() - dk / 2, k_native.max() + dk / 2,
                  l_native.min() - dl / 2, l_native.max() + dl / 2]

    # symmetric integer ticks (-kmax, 0, kmax) in units of 1e-4 m^-1
    kt = np.floor(ext[1] * 1e4)
    lt = np.floor(ext[3] * 1e4)
    xticks = np.array([-kt, 0, kt]) * 1e-4
    yticks = np.array([-lt, 0, lt]) * 1e-4
    fmt = mticker.FuncFormatter(lambda x, p: f"{int(round(x * 1e4))}")

    fig = plt.figure(figsize=(1.0 * n_top + 1.2, 3.8), constrained_layout=True)
    gs = GridSpec(2, n_top + 1, figure=fig, width_ratios=[1] * n_top + [0.25])

    # invisible axes spanning the whole panel grid -> centered shared labels
    bg = fig.add_subplot(gs[:, :n_top])
    bg.set_frame_on(False)
    bg.set_xticks([]); bg.set_yticks([])
    bg.set_xlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize, labelpad=28)
    bg.set_ylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize, labelpad=34)

    def style(ax, show_x, show_y):
        ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
        ax.set_aspect("equal")
        ax.axhline(0, color="k", lw=0.5, alpha=0.3)
        ax.axvline(0, color="k", lw=0.5, alpha=0.3)
        ax.set_xticks(xticks); ax.set_yticks(yticks)
        ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt)
        ax.tick_params(labelsize=fontsize - 2,
                       labelbottom=show_x, labelleft=show_y)

    # top row: one panel per fine band
    for i in range(n_top):
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(top[i], origin="lower", extent=ext_native,
                       interpolation="nearest", cmap=cmap, norm=norm)
        lam = 2 * np.pi / np.sqrt(fine_edges[i] * fine_edges[i + 1]) / 1e3
        ax.set_title(rf"${lam:.1f}$ km", fontsize=fontsize - 2, pad=4)
        style(ax, show_x=False, show_y=(i == 0))
        if i == 0:
            ax.set_ylabel("Before", fontsize=fontsize, labelpad=6)

    # bottom row: each panel spans its n_sub parents and is centered under them
    for j in range(nm_c):
        ax = fig.add_subplot(gs[1, j * n_sub:(j + 1) * n_sub])
        ax.imshow(bottom[j], origin="lower", extent=ext,
                  interpolation="nearest", cmap=cmap, norm=norm)
        lam = 2 * np.pi / np.sqrt(m_edges[j] * m_edges[j + 1]) / 1e3
        ax.set_title(rf"$\lambda_z={lam:.1f}$ km", fontsize=fontsize - 2, pad=4)
        style(ax, show_x=True, show_y=True)
        if j == 0:
            ax.set_ylabel("After", fontsize=fontsize, labelpad=6)

    cax = fig.add_subplot(gs[:, n_top])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(r"$\log(1+\mathcal{N})$", fontsize=fontsize)
    cbar.ax.tick_params(labelsize=fontsize - 1)

    for e in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{e}",
                    dpi=400, bbox_inches="tight")
    plt.show()
# usage
t, tri = 0, 0
plot_before_after_coarsegrain(
    np.real(np.nan_to_num(N_cgz[t, tri], nan=0.0)), np.real(np.nan_to_num(m_plus[t, tri], nan=0.0)), k[tri], l[tri],
    N_coarse_grain[t, tri], k_edges, l_edges, m_edges,
    savename="coarse_graining_before_after", n_sub=2,
)